In [51]:
from typing import List, Callable, Iterable, Tuple
import math
import operator
import random

In [52]:
Tensor = list
# 1. Definimos la función sigmoide si no se puede importar
def sigmoid(x: float) -> float:
    return 1 / (1 + math.exp(-x))

# Definición de la aproximación de la inversa de la CDF normal
def inverse_normal_cdf(p: float, mu: float = 0, sigma: float = 1) -> float:
    """Si no es estándar, computa el estándar y escala"""
    if mu != 0 or sigma != 1:
        return mu + sigma * inverse_normal_cdf(p)

    low_p, low_z = 0.0, -10.0  # normal_cdf(-10) es casi 0
    hi_p, hi_z = 1.0, 10.0     # normal_cdf(10) es casi 1
    
    while hi_z - low_z > 0.0001:
        mid_z = (low_z + hi_z) / 2
        mid_p = normal_cdf(mid_z)
        if mid_p < p:
            low_z, low_p = mid_z, mid_p
        else:
            hi_z, hi_p = mid_z, mid_p

    return mid_z

def normal_cdf(x: float, mu: float = 0, sigma: float = 1) -> float:
    return (1 + math.erf((x - mu) / math.sqrt(2) / sigma)) / 2

Vector = List[float]

def dot(v: Vector, w: Vector) -> float:
    """Computa v_1 * w_1 + ... + v_n * w_n"""
    assert len(v) == len(w), "Los vectores deben tener la misma longitud"
    return sum(v_i * w_i for v_i, w_i in zip(v, w))


In [53]:
def shape(tensor: Tensor) -> List[int]:
    sizes: List[int] = []
    while isinstance(tensor, list):
        sizes.append(len(tensor))
        tensor = tensor[0]
    return sizes

assert shape([1, 2, 3]) == [3]
assert shape([[1, 2], [3, 4], [5, 6]]) == [3, 2]

In [54]:
def is_1d(tensor: Tensor) -> bool:
    """Si el tensor[0] es una lista, es un tensor de orden superior. De lo contrario, el tensor es 1-dimensional (es decir, un vector)."""
    return not isinstance(tensor[0], list)

assert is_1d([1, 2, 3])
assert not is_1d([[1, 2], [3, 4]])

In [55]:
def tensor_sum(tensor: Tensor) -> float:
    """Suma todos los valores en el tensor"""
    if is_1d(tensor):
        return sum(tensor)
    else:
        return sum(tensor_sum(tensor_i) for tensor_i in tensor)
    
assert tensor_sum([1, 2, 3]) == 6
assert tensor_sum([[1, 2], [3, 4]]) == 10

In [56]:
def tensor_apply(f: Callable[[float], float], tensor: Tensor) -> Tensor:
    """Se aplica a los elementos"""
    if is_1d(tensor):
        return [f(x) for x in tensor]
    else:
        return[tensor_apply(f, tensor_i) for tensor_i in tensor]
    
assert tensor_apply(lambda x: x + 1, [1, 2, 3]) == [2, 3, 4]
assert tensor_apply(lambda x: 2 * x, [[1, 2], [3, 4]]) == [[2, 4], [6, 8]]

In [57]:
def zeros_like(tensor: Tensor) -> Tensor:
    return tensor_apply(lambda _: 0.0, tensor)

assert zeros_like([1, 2, 3]) == [0, 0, 0]
assert zeros_like([[1, 2], [3, 4]]) == [[0, 0], [0, 0]]

In [58]:
def tensor_combine(f: Callable[[float, float], float], t1: Tensor, t2: Tensor) -> Tensor:
    """Se aplica a los elementos correspondientes de t1 y t2"""
    if is_1d(t1):
        return [f(x, y) for x, y in zip(t1, t2)]
    else:
        return [tensor_combine(f, t1_i, t2_i) for t1_i, t2_i in zip(t1, t2)]
    
assert tensor_combine(operator.add, [1, 2, 3], [4, 5, 6]) == [5, 7, 9]
assert tensor_combine(operator.mul, [1, 2, 3], [4, 5, 6]) == [4, 10, 18]

In [59]:
class Layer:
    """Nuestras redes neuronales estarán compuestas por Capas, cada una de las cuales Sabe cómo hacer algún cálculo sobre sus entradas en el "forward" Direccionar y propagar gradientes en la dirección "hacia atrás"."""

    def forward(self, input):
        """Tenga en cuenta la falta de tipos. No vamos a ser prescriptivos Acerca de qué tipos de entradas pueden tomar las capas y qué tipos De los productos pueden regresar."""
        raise NotImplementedError

    def backward(self, gradient):
        """Del mismo modo, no vamos a ser prescriptivos sobre lo que El gradiente parece. Depende de ti el usuario asegurarse Que estás haciendo las cosas con sensatez."""
        raise NotImplementedError
    
    def params(self) -> Iterable[Tensor]:
        """Devuelve los parámetros de esta capa. La implementación por defecto No devuelve nada, de modo que si tienes una capa sin parámetros No tienes que implementar esto."""
        return ()
    
    def grads(self) -> Iterable[Tensor]:
        """Devuelve los gradientes, en el mismo orden que params()."""
        return ()

In [60]:
class Sigmoid(Layer):
    def forward(self, input: Tensor) -> Tensor:
        self.sigmoids = tensor_apply(sigmoid, input)
        return self.sigmoids
    
    def backward(self, gradient: Tensor) -> Tensor:
        # La derivada de la sigmoide es: f(x) * (1 - f(x))
        return tensor_combine(lambda sig, grad: sig * (1 - sig) * grad, self.sigmoids, gradient)

In [61]:
def random_uniform(*dims: int) -> Tensor:
    if len(dims) == 1:
        return [random.random() for _ in range(dims[0])]
    else:
        return [random_uniform(*dims[1:]) for _ in range(dims[0])]
    
def random_normal(*dims: int, mean: float = 0.0, variance: float = 1.0) -> Tensor:
    if len(dims) == 1:
        return[mean + variance * inverse_normal_cdf(random.random()) for _ in range(dims[0])]
    else:
        return [random_normal(*dims[1:], mean=mean, variance=variance) for _ in range(dims[0])]
    
assert shape(random_uniform(2, 3, 4)) == [2, 3, 4]
assert shape(random_normal(5, 6, mean=10)) == [5, 6]   

In [62]:
def random_tensor(*dims: int, init: str = 'normal') -> Tensor:
    if init == 'normal':
        return random_normal(*dims)
    elif init == 'uniform':
        return random_uniform(*dims)
    elif init == 'xavier':
        # La lógica Xavier: varianza depende de la suma de dimensiones
        # Asegúrate de pasar variance= como argumento nombrado
        variance = 2.0 / sum(dims) 
        return random_normal(*dims, variance=variance) # <--- Cambiado aquí
    else:
        raise ValueError(f"unknown init: {init}")

In [63]:
class Linear(Layer):
    def __init__(self, input_dim: int, output_dim: int, init: str = 'xavier') -> None:
        """Una capa de neuronas output_dim, cada una con pesos input_dim (y un sesgo)."""
        self.input_dim = input_dim
        self.output_dim = output_dim
        # self.w[o] es los pesos para la neurona o
        self.w = random_tensor(output_dim, input_dim, init=init)
        # self.b[o] es del termino de sesgo para la neuron o
        self.b = random_tensor(output_dim, init=init)

    def forward(self, input: Tensor) -> Tensor:
        # Save the input to use in the backward pass.
        self.input = input

        # Return the vector of neuron outputs.
        return [dot(input, self.w[o]) + self.b[o] for o in range(self.output_dim)]

    def backward(self, gradient: Tensor) -> Tensor:
        # Each b[o] gets added to output[o], which means
        # the gradient of b is the same as the output gradient.
        self.b_grad = gradient

        # Each w[o][i] multiplies input[i] and gets added to output[o].
        # So its gradient is input[i] * gradient[o].
        self.w_grad = [[self.input[i] * gradient[o] for i in range(self.input_dim)] for o in range(self.output_dim)]

        # Each input[i] multiplies every w[o][i] and gets added to every
        # output[o]. So its gradient is the sum of w[o][i] * gradient[o]
        # across all the outputs.
        return [sum(self.w[o][i] * gradient[o] for o in range(self.output_dim)) for i in range(self.input_dim)]

    def params(self) -> Iterable[Tensor]:
        return [self.w, self.b]

    def grads(self) -> Iterable[Tensor]:
        return [self.w_grad, self.b_grad]


In [64]:
class Sequential(Layer):
    """Una capa que consiste en una secuencia de otras capas. Depende de usted asegurarse de que la salida de cada capa Tiene sentido como la entrada a la siguiente capa."""

    def __init__(self, layers: List[Layer]) -> None:
        self.layers = layers
    
    def forward(self, input):
        """Simplemente reenvía la entrada a través de las capas en orden."""
        for layer in self.layers:
            input = layer.forward(input)
        return input
    
    def backward(self, gradient):
        """Simplemente retropropague el gradiente a través de las capas en reversa"""
        for layer in reversed(self.layers):
            gradient = layer.backward(gradient)
        return gradient
    
    def params(self) -> Iterable[Tensor]:
        """Solo devuelve los párametros de cada capa."""
        return (param for layer in self.layers for param in layer.params())
    
    def grads(self) -> Iterable[Tensor]:
        """Solo devuelva los grados de cada capa."""
        return (grad for layer in self.layers for grad in layer.grads())

In [65]:
xor_net = Sequential([Linear(input_dim= 2, output_dim= 2), Sigmoid(), Linear(input_dim=2, output_dim=1), Sigmoid()])